# 1. Konfiguracja środowiska oraz datasetu

## 1.1. Instalacja zależności

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
%pip install -r requirements.txt

In [ ]:
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

## 1.2.Konfiguracja importów

In [ ]:
import torch
import random
from pathlib import Path
from IPython.display import Audio
from src.process_guitarset import process_dataset
from src.config import *
from src.dataset import load_all_batches, GuitarSetDataset
from src.visualization import (
    plot_audio_waveform,
    plot_cqt,
    plot_annotation_map,
    create_annotation_maps
)


## 1.3. Konfiguracja GPU - automatyczne wykrywanie

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Używane urządzenie: {device}")
if device.type == 'cuda':
    print(f"Model karty GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

## 1.4. Wczytanie guitarsetu

In [ ]:
config = {
    "data_dir": "src/guitarset_data",
    "output_dir": "src/processed_data",
    "batch_size": 32,
    "seed": 42,
    "apply_augmentation": False,
    "max_tracks": None,
    "overwrite": False
}

process_dataset(**config)


## 1.5. Wczytanie guitarseta z plików batch

In [ ]:
# Wczytanie danych z plików
train_dir = Path(config["output_dir"]) / "train"
val_dir = Path(config["output_dir"]) / "val"
test_dir = Path(config["output_dir"]) / "test"

train_data = load_all_batches(train_dir)
val_data = load_all_batches(val_dir)
test_data = load_all_batches(test_dir)

# Stworzenie Datasetów
train_dataset = GuitarSetDataset(train_data)
val_dataset = GuitarSetDataset(val_data)
test_dataset = GuitarSetDataset(test_data)

## 1.6. Wizualizacja przykładowego pliku

In [ ]:
sample = random.choice(train_data)

# Dane surowe
audio = sample["audio"].numpy()
cqt = sample["features"].numpy()
sr = sample["sample_rate"]

# Wizualizacja audio i CQT
plot_audio_waveform(audio, sr)
plot_cqt(cqt, sr, hop_length=FFT_HOP, bins_per_octave=12 * NOTES_BINS_PER_SEMITONE)

# Wizualizacja adnotacji nut i konturów
notes_map, contours_map = create_annotation_maps(sample)
plot_annotation_map(notes_map, title="Adnotacje nut (sparse piano roll)", ylabel="Bin częstotliwości (nuty)",
                    cmap='hot')
plot_annotation_map(contours_map, title="Adnotacje konturów (multif0)", ylabel="Bin częstotliwości (kontury)",
                    cmap='Blues')
# Odtwarzanie audio
Audio(audio, rate=sr)
